# Proyecto sobre Redes Neuronales Artificiales (Bonificación opcional)

### Redes Neuronales Artificiales
### Feed-Forward Neural Networks - Perceptrón Multicapa (MLP)

### Universidad de Medellin

### Facultad de Ingenierías

### Ingeniería de Sistemas

### UdeM

#### Profesor: Antonio Tamayo Herrera

## Guía de la práctica

En esta archivo va a encontrar tanto celdas de código cómo celdas de texto con las instrucciones para desarrollar el laboratorio.

Lea atentamente las instrucciones entregadas en las celdas de texto correspondientes y proceda con la solución de las preguntas planteadas.

Nota: no olvide ir ejecutando las celdas de código de arriba hacia abajo para que no tenga errores de importación de librerías o por falta de definición de variables.

## Indicaciones

Este ejercicio tiene como objetivo implementar una red neuronal artificial bajo la arquitectura feed-forward neural network o perceptrón multicapa (MLP) para resolver un problema de clasificación multiclase.

Para la implementación del modelo feed-forward neural network haremos uso de la librería keras y tensorflow. (Para instalarla use: !pip install keras y !pip install tensorflow, encontrará las celdas con este código más adelante, descomente y ejecute, la instalación puede tardar varios minutos). Si lo prefiere, puede desarrollar el laboratorio en google colaboratory y no necesitará instalar ninguna libreríía.

Para el problema de clasificación usaremos la siguiente base de datos: https://archive.ics.uci.edu/ml/datasets/Cardiotocography

#### Abstract:
The dataset consists of measurements of fetal heart rate (FHR) and uterine contraction (UC) features on cardiotocograms classified by expert obstetricians.


#### Data Set Information:

2126 fetal cardiotocograms (CTGs) were automatically processed and the respective diagnostic features measured. The CTGs were also classified by three expert obstetricians and a consensus classification label assigned to each of them. Classification was both with respect to a morphologic pattern (A, B, C. ...) and to a fetal state (N, S, P). Therefore the dataset can be used either for 10-class or 3-class experiments.

#### Attribute Information:

LB - FHR baseline (beats per minute)

AC - # of accelerations per second

FM - # of fetal movements per second

UC - # of uterine contractions per second

DL - # of light decelerations per second

DS - # of severe decelerations per second

DP - # of prolongued decelerations per second

ASTV - percentage of time with abnormal short term variability

MSTV - mean value of short term variability

ALTV - percentage of time with abnormal long term variability

MLTV - mean value of long term variability

Width - width of FHR histogram

Min - minimum of FHR histogram

Max - Maximum of FHR histogram

Nmax - # of histogram peaks

Nzeros - # of histogram zeros

Mode - histogram mode

Mean - histogram mean

Median - histogram median

Variance - histogram variance

Tendency - histogram tendency

CLASS - FHR pattern class code (1 to 10)

NSP - fetal state class code (N=normal (1); S=suspect (2); P=pathologic (3))

En la siguiente celda de código se realiza la transformación de la variable de salida Y al formato One Hot Encoding.

In [ ]:
!pip install keras

In [ ]:
!pip install tensorflow

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
from __future__ import division

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder
import time

def oneHotEncoder(y):
    # Se cambia 'sparse' por 'sparse_output' para compatibilidad con las versiones más recientes de scikit-learn
    onehot_encoder = OneHotEncoder(sparse_output=False)
    integer_encoded = y.reshape(len(y), 1)
    onehot_encoded = onehot_encoder.fit_transform(integer_encoded)
    return onehot_encoded

#cargamos la bd de entrenamiento
db = np.loadtxt('/content/gdrive/MyDrive/Aprendizaje de Máquina/Proyecto de bonificación sobre RNA/DB_Fetal_Cardiotocograms.txt')  # Assuming tab-delimiter

# Usamos la matriz 'db' de NumPy cargada desde el archivo txt
# El dataset original tiene 22 columnas (índices 0 a 21).
# Las características van desde la columna 0 hasta la 20 (21 características en total)
X = db[:, 0:21]

# La etiqueta NSP está en la última columna (índice 21)
Y = db[:, 22]
Y = oneHotEncoder(Y)

print(Y)
print(X)

## Ejercicio 1.

1.1. Por qué es necesario hacer la codificación One-Hot Encodign de la variable de salida?

R/: Para poder supervisar el modelo
    
1.2 De qué manera se toma la decisión para asignar la clase una muestra en la red feed-forward cuando hay múltiples clases?

R/: con la función Softmax se escoge la de mayor probalidad

La siguiente celda tiene un método para calcular las medidas de error, segúún sea un problema biclase o multiclase, solo debe ejecutarla.

In [ ]:
def metrics(y_test, y_pred):
  '''
  Parametros:
    - lista (array) y_test (One-hot encoding)
    - lista (array) y_predichas (softmax vector)
  Return:
    - F1
    - Recall
    - Precision
    - Accuracy
    - Especificity
  '''
  y_test = np.asarray(y_test)
  y_pred = np.asarray(y_pred)

  if y_test[0].shape[0] == 1:    #Biclase
    TP,TN,FP,FN = 0,0,0,0
    for t,p in zip(y_test,y_pred):
      if t==1:
        if p>=0.5: #yp==1:
          TP+=1
        else:
          FN+=1
      else:
        if p>=0.5: #yp==1:
          FP+=1
        else:
          TN+=1

    recall = TP/(TP+FN)
    prec = TP/(TP+FP)
    acc = (TP+TN)/(TP+FP+FN+TN)
    f1 = (2*recall*prec)/(recall+prec)
    esp = TN/(TN+FP)

    return f1, recall, prec, acc, esp

  else:    #Multiclase
    error = 0
    for test,pred in zip(y_test,y_pred):
        t = np.argmax(test)
        p = np.argmax(pred)
        if t!=p:
          error+=1

    err = error/np.shape(y_test)[0]
    acc = 1 - err

    return acc


## Ejercicio 2

En la siguiente celda, complete el código donde le sea indicado. Consulte la documentación oficial de la librería keras para la arquitectura propuesta aquí: https://keras.io/getting-started/sequential-model-guide/

In [ ]:
import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation

#Para calcular el costo computacional
tiempo_i = time.time()

input_dim = np.shape(X)[1]
output_dim = np.shape(Y)[1]

#Complete el código para crear la red neuronal artificial.
#Para la configuración de la red observe la tabla de más abajo.
# Creación de la red neuronal artificial para el primer experimento de la tabla:
# Capa oculta con 10 neuronas (activación relu) y capa de salida con 3 neuronas (activación softmax)
model = Sequential()
model.add(Dense(10, input_dim= input_dim, activation='relu'))
model.add(Dense(2, input_dim= input_dim, activation='relu'))
model.add(Dense(output_dim, activation='softmax'))

#Implemetamos la metodología de validación cross validation con 10 folds
Accuracy = np.zeros(10)

kf = KFold(n_splits=10, shuffle=True)
j = 0
for train_index, test_index in kf.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = Y[train_index], Y[test_index]

    # Compilamos el modelo con el optimizador Adam y la pérdida categorical_crossentropy
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    # Entrenamos el modelo con 10 épocas y desactivamos la salida detallada con verbose=0 para limpieza
    model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0)

    # Validación del modelo
    y_pred = model.predict(X_test, verbose=0)

    Accuracy[j] = metrics(y_test, y_pred)
    j+=1

print("\nEficiencia en la clasificacióón: " + str(np.mean(Accuracy)) + " +/- " + str(np.std(Accuracy)))

print ("\n\nTiempo total de ejecución: " + str(time.time()-tiempo_i)+ " segundos.")

2.1 Varíe el número de épocas del algoritmo de entrenamiento y el número de neuronas de la capa oculta y complete la siguiente tabla.

<table>
  <tr>
    <th># de Epocas</th>
    <th># Neuronas en 1ra capa oculta</th>
    <th># Neuronas en 2da capa oculta</th>
    <th>Eficiencia +/- IC</th>
    <th>Tiempo de ejecución</th>
  </tr>
  <tr>
    <td>10</td>
    <td>10</td>
    <td>N/A</td>
    <td>0.8589002568872353 +/- 0.027171698546283075</td>
    <td>37.8 Segundos</td>
  </tr>
  <tr>
    <td>30</td>
    <td>10</td>
    <td>N/A</td>
    <td>0.8622420054920719 +/- 0.04809190687070046</td>
    <td>71.7 Segundos</td>
  </tr>
  <tr>
    <td>10</td>
    <td>10</td>
    <td>2</td>
    <td>0.778452475861458 +/- 0.026993429911749203</td>
    <td>40.88 Segundos</td>
  </tr>
  <tr>
    <td>30</td>
    <td>10</td>
    <td>2</td>
    <td>0.7784679776773851 +/- 0.025263197075570534</td>
    <td>78.78 Segundos</td>
  </tr>
  <tr>
    <td>50</td>
    <td>10</td>
    <td>2</td>
    <td>0.7784236867747365 +/- 0.027402631940845173</td>
    <td>116.05 Segundos</td>
  </tr>
</table>


2.2 Analice e interprete los resultados de la tabla anterior. ¿Cómo influye el cambio de los parámetros propuesto en la eficiencia y en el tiempo de ejecución? Explique.

R/:Al observar los resultados, se evidencia que aumentar de 10 a 30 épocas con una sola capa oculta de 10 neuronas mejora ligeramente la eficiencia, de aproximadamente 85,9 % a 86,2 %, aunque el tiempo de ejecución se duplica, pasando de 38 a 72 segundos. Por otro lado, agregar una segunda capa oculta de solo 2 neuronas reduce la eficiencia al 77,8 %, independientemente del número de épocas, esto indica que las 2 neuronas de la segunda capa no fueron suficientes para procesar toda la información correctamente. Además, por más que se aumentaran las épocas no mejoró y el tiempo aumentaba. En conclusión, el número de capas y neuronas tiene mayor impacto que el número de épocas.

## Ejercicio 3

Ahora va a resolver el problema de regresión de la base de datos AirQuality (https://archive.ics.uci.edu/ml/datasets/Air+Quality) usando RNA bajo la arquitectura feed-forward. En la siguiente celda no tiene que implementar código, solo ejecutarla y analizar que proceso de limpieza que se lleva a cabo.

In [ ]:
import numpy as np

#cargamos la bd de entrenamiento
db = np.loadtxt('/content/gdrive/MyDrive/Aprendizaje de Máquina/Proyecto de bonificación sobre RNA/AirQuality.data')  # Assuming tab-delimiter
print ("Dim de la base de datos original: " + str(np.shape(db)))
db = db.reshape(9357,13)
DataBase = db
#La base d edatos AirQuality tiene muchos valores perdidos, vamos a quitar todos esos registros de la base de datos
#Están marcados como -200. Donde haya un valor -200 eliminaremos el registro.
j = 0
for i in range(0,np.size(db,0)):
    if -200 == db[i,12]:
        #print i
        j+=1
        DataBase = np.delete(DataBase,i,0)

print ("\nHay " + str(j) + " valores perdidos en la variable de salida. :(")

print ("\nDim de la base de datos sin las muestras con variable de salida perdido "+ str(np.shape(DataBase)))

#Ya hemos eliminado los registros con valor de la variable de salida perdido.

#Ahora vamos a imputar los valores perdidos en cada una de las características
print ("\nProcesando imputación de valores perdidos en las características . . .\n")

#data = open("AirQuality_cleaned.data", "w")

for k in range(0,np.size(DataBase,0)):
    for w in range(0,13):
        if -200 == DataBase[k,w]:
            DataBase[k,w] = round(np.mean(DataBase[:,w]))

    #data.write(str(DataBase[k,:]))

#data.close()
print ("Imputación finalizada.\n")

hay_missed_values = False
for i in range(0,np.size(DataBase,0)):
    if -200 in DataBase[i,:]:
        hay_missed_values = True
if(hay_missed_values):
    print ("Hay valores perdidos")
else:
    print ("No hay valores perdidos en la base de datos. Ahora se puede procesar. La base de datos está en la variable DataBase")

In [ ]:
X = DataBase[:,0:12]

Y = DataBase[:,12]*10 #(se multiplica por 100 para evitar errores en la interpretación de la eficiencia del modelo porque
                      #los valores de Y menores que 1 pueden llevar a malas interpretaciones).
Y = Y.reshape(np.size(X,0),1)

In [ ]:
import keras
from keras.models import Sequential
from keras.layers import Dense, Input
from sklearn.model_selection import KFold
import time
import numpy as np

#Error cuadrático medio (para problemas de regresión)
def ECM(Y_real,Y_est):
    N = np.size(Y_real)
    ecm = sum((Y_est - Y_real)**2)/N
    return ecm

def crear_modelo_reg(input_dim, neuronas):
    model = Sequential()
    model.add(Input(shape=(input_dim,)))
    model.add(Dense(neuronas, activation='relu'))  # Capa oculta
    model.add(Dense(1, activation='linear'))       # Capa de salida: función identidad
    model.compile(loss='mean_absolute_error', optimizer='adam', metrics=['mean_squared_error'])
    return model

def evaluar_regresion(X, Y, epocas, neuronas, verbose=0):
    #Para calcular el costo computacional
    tiempo_i = time.time()

    #Implementamos la metodología de validación cross validation con 10 folds
    Errores = np.ones(10)
    j = 0
    kf = KFold(n_splits=10)

    # Guardamos las predicciones de cada fold para poder graficarlas después
    y_real_total = np.zeros_like(Y)
    y_pred_total = np.zeros_like(Y)

    for train_index, test_index in kf.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = Y[train_index], Y[test_index]

        # Modelo nuevo en cada fold (pesos reiniciados), compilado con el optimizador Adam
        model = crear_modelo_reg(X.shape[1], neuronas)

        # Entrenamiento del modelo
        model.fit(X_train, y_train, epochs=epocas, batch_size=100, validation_split=0.1, verbose=verbose)

        # Validación del modelo
        y_pred = model.predict(X_test, verbose=0)

        Errores[j] = ECM(y_test, y_pred)[0]
        y_real_total[test_index] = y_test
        y_pred_total[test_index] = y_pred
        j+=1

    t = time.time()-tiempo_i
    print(f"Épocas={epocas}, neuronas en capa oculta={neuronas}")
    print("Error de validación: " + str(np.mean(Errores)) + " +/- " + str(np.std(Errores)))
    print("Tiempo total de ejecución: " + str(t) + " segundos.\n")
    return np.mean(Errores), np.std(Errores), t, y_real_total, y_pred_total

# Ejecución individual (cambie los parámetros según la fila de la tabla)
_ = evaluar_regresion(X, Y, epocas=100, neuronas=10)

3.1 Defina en sus palabras que es una época en el modelo de redes neuronales artificiales.

R/: Una época es cuando pasan todos los datos de entrenamiento a través de la red neuronal, cada vez que pasa un dato se van actualizando los pesos (backpropagation)

3.2 Por qué se usa la función de activación lineal en lugar de reLu, tanh o sigmoid en la capa de salida para este problema?

R/: Porque es un problema de regresión y la salida tiene que ser un número continuo que puede tomar cualquier valor. Sigmoid y tanh limitan la salida a un rango (0 a 1 o -1 a 1) y ReLU no deja valores negativos, en cambio la lineal deja pasar el valor tal cual.




3.3 Varíe el número de épocas del algoritmo de entrenamiento y el número de neuronas de la capa oculta y complete la siguiente tabla.

<table>
  <tr>
    <th># de Epocas</th>
    <th># Neuronas en capa oculta</th>
    <th>ECM +/- IC</th>
    <th>Tiempo de ejecución</th>
  </tr>
  <tr>
    <td>10</td>
    <td>10</td>
    <td>112.72899114235186 +/- 87.51555843528007</td>
    <td>51.42 Segundos</td>
  </tr>
  <tr>
    <td>10</td>
    <td>20</td>
    <td>127.58843286067209 +/- 110.53445175004009</td>
    <td>51.26 Segundos</td>
  </tr>
  <tr>
    <td>100</td>
    <td>10</td>
    <td>68.1226143284451 +/- 80.25982569071236</td>
    <td>294.72 Segundos</td>
  </tr>
</table>


3.4 Para esta base de datos con casi 10 mil registros, que puede decir del costo computacional del modelo con 100 épocas y 10 neuronas en la capa oculta

R/: El costo compitacional es significativamente alto, con casi 10 mil registros y 100 épocas, el modelo tuvo que procesar todo el conjunto 100 veces lo que resultó en casi 5 minutos, pero también hay que resaltar que el EMC disminuyó a casi la mitad, lo que nos dice que el aumento de épocas si valió la pena.

3.5 Haga un plot de las salidas reales y las predicciones por la RNA en la misma gráfica, para evaluar visualmente el desempeño de la red. Esto solo para la configuración en la cuál se encontró el menor error.

In [ ]:
import matplotlib.pyplot as plt

# La mejor configuración según la tabla es 100 épocas y 10 neuronas en la capa oculta.
# Ejecutamos la evaluación para obtener las predicciones reales y estimadas directas.
epocas_mejor = 100
neuronas_mejor = 10

print("Entrenando y evaluando la mejor configuración para generar la gráfica...")
_, _, _, y_real, y_est = evaluar_regresion(X, Y, epocas=epocas_mejor, neuronas=neuronas_mejor, verbose=0)

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# 1) Serie completa: salidas reales vs predicciones (predicciones de validación cruzada)
axes[0].plot(y_real, label='Salida real', color='tab:blue', linewidth=0.8)
axes[0].plot(y_est, label='Predicción RNA', color='tab:orange', linewidth=0.8, alpha=0.8)
axes[0].set_title(f'Salidas reales vs predicciones RNA ({epocas_mejor} épocas, {neuronas_mejor} neuronas)')
axes[0].set_xlabel('Muestra')
axes[0].set_ylabel('Humedad absoluta (x10)')
axes[0].legend()

# 2) Acercamiento a 300 muestras para ver mejor el ajuste
axes[1].plot(y_real[:300], 'o-', label='Salida real', markersize=3)
axes[1].plot(y_est[:300], 's-', label='Predicción RNA', markersize=3, alpha=0.8)
axes[1].set_title('Acercamiento: primeras 300 muestras')
axes[1].set_xlabel('Muestra')
axes[1].set_ylabel('Humedad absoluta (x10)')
axes[1].legend()

plt.tight_layout()
plt.show()